# Breast Cancer Classification Project

## Project Overview

This project uses a real breast cancer diagnostic dataset to build and compare several supervised machine learning classification models. The goal is to predict whether a tumor is **malignant** or **benign** based on numeric measurements computed from digitized images of breast mass cell nuclei.

## Dataset Description

The dataset contains **569 observations** and **30 numeric predictor variables**. These predictors describe characteristics of cell nuclei, such as radius, texture, perimeter, area, smoothness, compactness, concavity, symmetry, and fractal dimension.

The target variable is:

- `diagnosis = 0`: malignant tumor
- `diagnosis = 1`: benign tumor

A second target column, `diagnosis_label`, provides the readable class labels.

## Project Objective

The objective is to train and evaluate multiple classification models, including Logistic Regression, KNN, Decision Tree, Random Forest, Naive Bayes, and SVM. The models are compared using common classification metrics such as accuracy, precision, recall, F1 score, ROC AUC, confusion matrix, and precision recall curves.

## Brief Description of the Classification Algorithms

### 1. Logistic Regression

Logistic Regression estimates the probability that an observation belongs to a particular class.

**Strengths**

- Easy to interpret
- Fast to train
- Works well as a baseline model
- Performs well when the relationship between features and the outcome is approximately linear

**Weaknesses**

- May perform poorly when the relationship is highly nonlinear
- Sensitive to multicollinearity among predictors
- Usually requires feature scaling for better performance

---

### 2. K Nearest Neighbors

K-Nearest Neighbors classifies a new observation based on the classes of the k closest observations in the training data.

**Strengths**

- Simple and intuitive
- Can capture nonlinear patterns
- Makes a few assumptions about the data

**Weaknesses**

- Can be slow with large datasets
- Very sensitive to feature scaling
- Performance depends heavily on the choice of `k`
- Can be affected by noisy or irrelevant features

---

### 3. Decision Tree

A Decision Tree makes predictions by splitting the data into smaller groups based on feature values.

**Strengths**

- Easy to understand and visualize
- Can capture nonlinear relationships
- Does not require feature scaling
- Handles interactions between variables well

**Weaknesses**

- Can easily overfit the training data
- Small changes in the data can produce a very different tree
- Often less accurate than ensemble tree methods such as Random Forest

---

### 4. Random Forest

Random Forest builds many decision trees and combines their predictions to improve accuracy and reduce overfitting.

**Strengths**

- Usually performs very well
- Reduces overfitting compared with a single decision tree
- Handles nonlinear relationships and feature interactions
- Provides feature importance scores
- Does not require feature scaling

**Weaknesses**

- Less interpretable than a single decision tree
- Can be slower to train than simpler models
- May require tuning for best performance

---

### 5. Naive Bayes

Naive Bayes is a probabilistic classifier based on Bayes' theorem. It assumes that features are conditionally independent given the class.

**Strengths**

- Very fast and efficient
- Works well with small datasets
- Often performs well as a simple benchmark model
- Commonly used in text classification

**Weaknesses**

- The independence assumption is often unrealistic
- May perform poorly when predictors are strongly correlated
- Can be less accurate than more flexible models

---

### 6. Support Vector Machine

Support Vector Machine finds a decision boundary that best separates the classes. With kernels, it can capture nonlinear patterns.

**Strengths**

- Effective in high dimensional datasets
- Can model nonlinear relationships using kernels
- Often performs well when classes are clearly separable
- Robust when properly tuned

**Weaknesses**

- Requires feature scaling
- Can be slow on large datasets
- Less interpretable than Logistic Regression or Decision Trees
- Performance depends on hyperparameters such as `C`, `gamma`, and kernel choice

### Import Libraries

In [11]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    log_loss,
    confusion_matrix,
    classification_report,
    roc_curve,
    precision_recall_curve
)

### Import data

In [12]:
df = pd.read_csv("../data/breast_cancer_classification.csv")

df.head()

,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension,diagnosis,diagnosis_label
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,...,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,0,malignant
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,...,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,0,malignant
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,...,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,0,malignant
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,0.09744,...,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,0,malignant
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,...,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,0,malignant


In [13]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 569 entries, 0 to 568
Data columns (total 32 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   mean radius              569 non-null    float64
 1   mean texture             569 non-null    float64
 2   mean perimeter           569 non-null    float64
 3   mean area                569 non-null    float64
 4   mean smoothness          569 non-null    float64
 5   mean compactness         569 non-null    float64
 6   mean concavity           569 non-null    float64
 7   mean concave points      569 non-null    float64
 8   mean symmetry            569 non-null    float64
 9   mean fractal dimension   569 non-null    float64
 10  radius error             569 non-null    float64
 11  texture error            569 non-null    float64
 12  perimeter error          569 non-null    float64
 13  area error               569 non-null    float64
 14  smoothness error         569 non-null

In [14]:
df.describe()

,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension,diagnosis
count,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,...,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000
mean,14.127292,19.289649,91.969033,654.889104,0.096360,0.104341,0.088799,0.048919,0.181162,0.062798,...,25.677223,107.261213,880.583128,0.132369,0.254265,0.272188,0.114606,0.290076,0.083946,0.627417
std,3.524049,4.301036,24.298981,351.914129,0.014064,0.052813,0.079720,0.038803,0.027414,0.007060,...,6.146258,33.602542,569.356993,0.022832,0.157336,0.208624,0.065732,0.061867,0.018061,0.483918
min,6.981000,9.710000,43.790000,143.500000,0.052630,0.019380,0.000000,0.000000,0.106000,0.049960,...,12.020000,50.410000,185.200000,0.071170,0.027290,0.000000,0.000000,0.156500,0.055040,0.000000
25%,11.700000,16.170000,75.170000,420.300000,0.086370,0.064920,0.029560,0.020310,0.161900,0.057700,...,21.080000,84.110000,515.300000,0.116600,0.147200,0.114500,0.064930,0.250400,0.071460,0.000000
50%,13.370000,18.840000,86.240000,551.100000,0.095870,0.092630,0.061540,0.033500,0.179200,0.061540,...,25.410000,97.660000,686.500000,0.131300,0.211900,0.226700,0.099930,0.282200,0.080040,1.000000
75%,15.780000,21.800000,104.100000,782.700000,0.105300,0.130400,0.130700,0.074000,0.195700,0.066120,...,29.720000,125.400000,1084.000000,0.146000,0.339100,0.382900,0.161400,0.317900,0.092080,1.000000
max,28.110000,39.280000,188.500000,2501.000000,0.163400,0.345400,0.426800,0.201200,0.304000,0.097440,...,49.540000,251.200000,4254.000000,0.222600,1.058000,1.252000,0.291000,0.663800,0.207500,1.000000


### Split features and target

In [15]:
X = df.drop(columns=["diagnosis", "diagnosis_label"])
y = df["diagnosis"]

In [16]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

### Define models

In [17]:
models = {
    "Logistic Regression": Pipeline(
        steps=[
            ("scaler", StandardScaler()),
            ("model", LogisticRegression(max_iter=1000, random_state=42))]),
    
    "KNN": Pipeline(
        steps=[
            ("scaler", StandardScaler()),
            ("model", KNeighborsClassifier(n_neighbors=7))]),
    
    "Decision Tree": DecisionTreeClassifier(
        max_depth=5, min_samples_leaf=10, random_state=42),
    
    "Random Forest": RandomForestClassifier(
        n_estimators=300, min_samples_leaf=3, random_state=42),
    
    "Naive Bayes": GaussianNB(),
    
    "SVM": Pipeline(steps=[("scaler", StandardScaler()),
            ("model", SVC( kernel="rbf", C=1.0, gamma="scale", probability=True, random_state=42)) ])
}

### Evaluation function

In [18]:
def evaluate_classification_model(model_name, y_true, y_pred, y_proba):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    specificity = tn / (tn + fp)
    
    return {
        "Model": model_name,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred),
        "Recall": recall_score(y_true, y_pred),
        "Specificity": specificity,
        "F1 Score": f1_score(y_true, y_pred),
        "ROC AUC": roc_auc_score(y_true, y_proba),
        "Average Precision": average_precision_score(y_true, y_proba),
        "Log Loss": log_loss(y_true, y_proba)
    }

### Train and evaluate models

In [19]:
results_list = []
predictions = {}
probabilities = {}

for model_name, model in models.items():
    model.fit(X_train, y_train)
    
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    
    predictions[model_name] = y_pred
    probabilities[model_name] = y_proba
    
    results_list.append(
        evaluate_classification_model(
            model_name,
            y_test,
            y_pred,
            y_proba
        )
    )

results_df = pd.DataFrame(results_list)

In [28]:
# Sort results by ROC AUC
sorted_results = results_df.sort_values(by="ROC AUC", ascending=False)

print("Test Set Model Performance")
print("--------------------------")
print(sorted_results)

# Get the best two model names
best_model_1 = sorted_results.iloc[0]["Model"]
best_model_2 = sorted_results.iloc[1]["Model"]

print(f"\n{best_model_1} and {best_model_2} are the best overall models.")

Test Set Model Performance
--------------------------
                 Model  Accuracy  Precision    Recall  Specificity  F1 Score  \
0  Logistic Regression  0.982456   0.986111  0.986111     0.976190  0.986111   
5                  SVM  0.982456   0.986111  0.986111     0.976190  0.986111   
3        Random Forest  0.956140   0.958904  0.972222     0.928571  0.965517   
2        Decision Tree  0.947368   0.958333  0.958333     0.928571  0.958333   
1                  KNN  0.973684   0.960000  1.000000     0.928571  0.979592   
4          Naive Bayes  0.938596   0.945205  0.958333     0.904762  0.951724   

    ROC AUC  Average Precision  Log Loss  
0  0.995370           0.997065  0.077746  
5  0.995040           0.996931  0.086009  
3  0.992725           0.995755  0.115979  
2  0.990079           0.992448  0.124829  
1  0.988426           0.989458  0.107417  
4  0.987765           0.992640  0.371256  

Logistic Regression and SVM are the best overall models.


### Cross validation

Cross validation is used because one train test split can give a performance result that depends too much on how the data happened to be split. It tells us how stable the model is across different samples of the data. Is answers the question: "Is this model consistently good, or did it only perform well on one lucky test split?"

In [22]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [23]:
scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc"
}

In [24]:
cv_results_list = []

for model_name, model in models.items():
    cv_results = cross_validate(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring=scoring,
        return_train_score=False
    )
    
    cv_results_list.append({
        "Model": model_name,
        "CV Accuracy Mean": cv_results["test_accuracy"].mean(),
        "CV Accuracy Std": cv_results["test_accuracy"].std(),
        "CV Precision Mean": cv_results["test_precision"].mean(),
        "CV Recall Mean": cv_results["test_recall"].mean(),
        "CV F1 Mean": cv_results["test_f1"].mean(),
        "CV ROC AUC Mean": cv_results["test_roc_auc"].mean(),
        "CV ROC AUC Std": cv_results["test_roc_auc"].std()
    })

cv_results_df = pd.DataFrame(cv_results_list)

cv_results_df.head()

,Model,CV Accuracy Mean,CV Accuracy Std,CV Precision Mean,CV Recall Mean,CV F1 Mean,CV ROC AUC Mean,CV ROC AUC Std
0,Logistic Regression,0.978022,0.009829,0.979472,0.985965,0.982544,0.995872,0.004960
1,KNN,0.962637,0.020382,0.956236,0.985965,0.970665,0.988596,0.008370
2,Decision Tree,0.914286,0.025441,0.956305,0.905263,0.929099,0.960114,0.021663
3,Random Forest,0.958242,0.014579,0.965413,0.968421,0.966609,0.990402,0.006308
4,Naive Bayes,0.940659,0.024670,0.942185,0.964912,0.953280,0.987719,0.006144


### Final comparison table

In [29]:
final_comparison = results_df.merge(
    cv_results_df,
    on="Model",
    how="left"
)

final_comparison = final_comparison.sort_values(
    by="ROC AUC",
    ascending=False
)

print("\nFinal Model Comparison")
print("----------------------")
print(final_comparison)


Final Model Comparison
----------------------
                 Model  Accuracy  Precision    Recall  Specificity  F1 Score  \
0  Logistic Regression  0.982456   0.986111  0.986111     0.976190  0.986111   
5                  SVM  0.982456   0.986111  0.986111     0.976190  0.986111   
3        Random Forest  0.956140   0.958904  0.972222     0.928571  0.965517   
2        Decision Tree  0.947368   0.958333  0.958333     0.928571  0.958333   
1                  KNN  0.973684   0.960000  1.000000     0.928571  0.979592   
4          Naive Bayes  0.938596   0.945205  0.958333     0.904762  0.951724   

    ROC AUC  Average Precision  Log Loss  CV Accuracy Mean  CV Accuracy Std  \
0  0.995370           0.997065  0.077746          0.978022         0.009829   
5  0.995040           0.996931  0.086009          0.969231         0.014579   
3  0.992725           0.995755  0.115979          0.958242         0.014579   
2  0.990079           0.992448  0.124829          0.914286         0.025441 

### Classification reports

In [30]:
for model_name in models.keys():
    print("\n" + "=" * 60)
    print(model_name)
    print("=" * 60)
    print(classification_report(y_test, predictions[model_name]))


Logistic Regression
              precision    recall  f1-score   support

           0       0.98      0.98      0.98        42
           1       0.99      0.99      0.99        72

    accuracy                           0.98       114
   macro avg       0.98      0.98      0.98       114
weighted avg       0.98      0.98      0.98       114


KNN
              precision    recall  f1-score   support

           0       1.00      0.93      0.96        42
           1       0.96      1.00      0.98        72

    accuracy                           0.97       114
   macro avg       0.98      0.96      0.97       114
weighted avg       0.97      0.97      0.97       114


Decision Tree
              precision    recall  f1-score   support

           0       0.93      0.93      0.93        42
           1       0.96      0.96      0.96        72

    accuracy                           0.95       114
   macro avg       0.94      0.94      0.94       114
weighted avg       0.95      0.95

### Best model

In [31]:
best_model_name = final_comparison.iloc[0]["Model"]
best_model = models[best_model_name]

print("\nBest Model Based on Test ROC AUC")
print("--------------------------------")
print(best_model_name)


Best Model Based on Test ROC AUC
--------------------------------
Logistic Regression
